# Running-average SGD trajectories and PCA

This notebook trains a small MLP on a noisy XOR classification task for exactly 1,000 minibatch-SGD updates on **`cuda:7`**.

Let $w_i$ be the flattened parameter vector immediately **after** update $i$, for $i=0,\ldots,999$. The causal trailing average is

$$\bar w_i = \frac{1}{i-s_i+1}\sum_{j=s_i}^{i}w_j, \qquad s_i=\max(0,i-49).$$

Thus the first 49 points use the available expanding prefix, and every later point averages exactly 50 updates. PCA is fit only to $[\bar w_i]_{i=0}^{999}$. Both the averaged path and the raw, noisy SGD path are centered with the averaged-path PCA mean and projected into the same top-3 and top-2 bases.

In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

SEED = 1234
STEPS = 1_000
WINDOW = 50
BATCH_SIZE = 64
LEARNING_RATE = 0.05
OUT_DIR = Path("diff_plots_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    gpu_devices = jax.devices("gpu")
except RuntimeError as exc:
    raise RuntimeError("This notebook requires a CUDA-enabled JAX installation.") from exc

if len(gpu_devices) <= 7:
    raise RuntimeError(
        f"cuda:7 was requested, but only {len(gpu_devices)} GPU device(s) are visible."
    )
DEVICE = gpu_devices[7]
if DEVICE.id != 7:
    raise RuntimeError(f"Expected physical cuda:7, selected {DEVICE}.")

print(f"JAX {jax.__version__}")
print(f"Using {DEVICE} ({DEVICE.device_kind})")
print(f"Plots will be written to {OUT_DIR.resolve()}")

## Toy task and model

The dataset consists of four noisy Gaussian clusters with XOR labels. The model is a `2 → 16 → 16 → 2` tanh MLP (354 trainable scalars). Minibatches are sampled with replacement from a fixed random schedule, giving a reproducible noisy SGD trajectory.

In [ ]:
def init_layer(key, n_in, n_out):
    limit = np.sqrt(6.0 / (n_in + n_out))
    weights = jax.random.uniform(
        key, (n_in, n_out), minval=-limit, maxval=limit, dtype=jnp.float32
    )
    bias = jnp.zeros((n_out,), dtype=jnp.float32)
    return weights, bias


def init_mlp(key):
    key1, key2, key3 = jax.random.split(key, 3)
    w1, b1 = init_layer(key1, 2, 16)
    w2, b2 = init_layer(key2, 16, 16)
    w3, b3 = init_layer(key3, 16, 2)
    return (w1, b1, w2, b2, w3, b3)


def mlp_logits(params, inputs):
    w1, b1, w2, b2, w3, b3 = params
    hidden1 = jnp.tanh(inputs @ w1 + b1)
    hidden2 = jnp.tanh(hidden1 @ w2 + b2)
    return hidden2 @ w3 + b3


def cross_entropy(params, inputs, labels):
    log_probs = jax.nn.log_softmax(mlp_logits(params, inputs), axis=1)
    chosen = jnp.take_along_axis(log_probs, labels[:, None], axis=1)
    return -jnp.mean(chosen)


def flatten_params(params):
    return jnp.concatenate([array.reshape(-1) for array in params])


with jax.default_device(DEVICE):
    root_key = jax.random.key(SEED)
    data_key, perm_key, init_key, batch_key = jax.random.split(root_key, 4)

    centers = jnp.asarray(
        [[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]],
        dtype=jnp.float32,
    )
    center_labels = jnp.asarray([0, 1, 1, 0], dtype=jnp.int32)
    points_per_center = 256
    features = jnp.repeat(centers, points_per_center, axis=0)
    labels = jnp.repeat(center_labels, points_per_center, axis=0)
    features = features + 0.35 * jax.random.normal(
        data_key, features.shape, dtype=jnp.float32
    )
    permutation = jax.random.permutation(perm_key, features.shape[0])
    features = features[permutation]
    labels = labels[permutation]

    initial_params = init_mlp(init_key)
    batch_indices = jax.random.randint(
        batch_key,
        shape=(STEPS, BATCH_SIZE),
        minval=0,
        maxval=features.shape[0],
        dtype=jnp.int32,
    )

parameter_count = int(flatten_params(initial_params).size)
assert parameter_count == 354
assert features.devices() == {DEVICE}
assert batch_indices.devices() == {DEVICE}
print(f"Dataset: {features.shape[0]} points; model: {parameter_count} parameters")

## Exactly 1,000 minibatch-SGD updates

This is plain SGD: no momentum, adaptive statistics, or weight decay. `jax.lax.scan` keeps the entire optimization loop and all post-update snapshots on `cuda:7`.

In [ ]:
@jax.jit(device=DEVICE)
def run_sgd(params, all_features, all_labels, all_batch_indices):
    def update_once(current_params, indices):
        loss, gradients = jax.value_and_grad(cross_entropy)(
            current_params, all_features[indices], all_labels[indices]
        )
        next_params = jax.tree_util.tree_map(
            lambda parameter, gradient: parameter - LEARNING_RATE * gradient,
            current_params,
            gradients,
        )
        # Store w_i after optimizer update i.
        return next_params, (loss, flatten_params(next_params))

    return jax.lax.scan(update_once, params, all_batch_indices)


final_params, (losses, w_sgd) = run_sgd(
    initial_params, features, labels, batch_indices
)
w_sgd.block_until_ready()

train_predictions = jnp.argmax(mlp_logits(final_params, features), axis=1)
train_accuracy = jnp.mean(train_predictions == labels)

assert w_sgd.shape == (STEPS, parameter_count)
assert w_sgd.devices() == {DEVICE}
assert losses.devices() == {DEVICE}
print(f"Recorded raw SGD path: {w_sgd.shape} on {w_sgd.devices()}")
print(f"Initial minibatch loss: {float(losses[0]):.4f}")
print(f"Final minibatch loss:   {float(losses[-1]):.4f}")
print(f"Full toy-set accuracy:  {100.0 * float(train_accuracy):.2f}%")

## Trailing averages and shared PCA coordinates

The PCA center $\mu$ and components are computed from the averaged trajectory only. If $V_k$ contains its top $k$ right singular vectors, the two projected paths are

$$Z_{\mathrm{avg},k} = (W_{\mathrm{avg}}-\mu)V_k, \qquad Z_{\mathrm{sgd},k} = (W_{\mathrm{sgd}}-\mu)V_k.$$

In particular, the raw path is **not** recentered independently. The top-2 variant uses the first two directions from this same PCA fit.

In [ ]:
with jax.default_device(DEVICE):
    prefix_sums = jnp.concatenate(
        [
            jnp.zeros((1, parameter_count), dtype=w_sgd.dtype),
            jnp.cumsum(w_sgd, axis=0),
        ],
        axis=0,
    )
    ends = jnp.arange(1, STEPS + 1, dtype=jnp.int32)
    starts = jnp.maximum(0, ends - WINDOW)
    counts = (ends - starts).astype(w_sgd.dtype)[:, None]
    w_avg = (prefix_sums[ends] - prefix_sums[starts]) / counts

    pca_mean = jnp.mean(w_avg, axis=0)
    centered_avg = w_avg - pca_mean
    _, singular_values, vh = jnp.linalg.svd(centered_avg, full_matrices=False)
    explained_ratio = singular_values**2 / jnp.sum(singular_values**2)

    components_3 = vh[:3].T
    avg_3d = centered_avg @ components_3
    sgd_3d = (w_sgd - pca_mean) @ components_3

    # Explicit top-2 PCA variant, using the leading two directions of the same fit.
    components_2 = vh[:2].T
    avg_2d = centered_avg @ components_2
    sgd_2d = (w_sgd - pca_mean) @ components_2

avg_3d.block_until_ready()

assert w_avg.shape == w_sgd.shape == (STEPS, parameter_count)
assert w_avg.devices() == {DEVICE}
assert avg_3d.devices() == {DEVICE}
assert sgd_3d.devices() == {DEVICE}
np.testing.assert_allclose(
    np.asarray(jax.device_get(w_avg[0])),
    np.asarray(jax.device_get(w_sgd[0])),
    rtol=1e-5,
    atol=1e-6,
)
np.testing.assert_allclose(
    np.asarray(jax.device_get(w_avg[49])),
    np.asarray(jax.device_get(jnp.mean(w_sgd[:50], axis=0))),
    rtol=2e-4,
    atol=2e-5,
)
np.testing.assert_allclose(
    np.asarray(jax.device_get(w_avg[-1])),
    np.asarray(jax.device_get(jnp.mean(w_sgd[-50:], axis=0))),
    rtol=2e-4,
    atol=2e-5,
)
np.testing.assert_allclose(
    np.asarray(jax.device_get(avg_2d)),
    np.asarray(jax.device_get(avg_3d[:, :2])),
    rtol=1e-5,
    atol=1e-6,
)

top3_ratio = np.asarray(jax.device_get(explained_ratio[:3]))
print("Top-3 explained-variance ratios:", np.round(top3_ratio, 6))
print(f"Top-3 cumulative explained variance: {100.0 * top3_ratio.sum():.2f}%")

## Save 3D and 2D path comparisons

The raw SGD path is thin and translucent; the 50-step running-average path is thicker. Dots mark every 100 updates, while green/red markers identify the averaged path's start/end. Two diagnostic figures are also saved alongside the requested trajectory plots.

In [ ]:
sgd_3d_np, avg_3d_np, sgd_2d_np, avg_2d_np, losses_np, evr_np = [
    np.asarray(jax.device_get(array))
    for array in (sgd_3d, avg_3d, sgd_2d, avg_2d, losses, explained_ratio)
]
milestones = np.unique(np.r_[np.arange(0, STEPS, 100), STEPS - 1])


def pc_label(component_index):
    return f"PC{component_index + 1} ({100.0 * evr_np[component_index]:.2f}% var.)"


fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.plot(
    sgd_3d_np[:, 0], sgd_3d_np[:, 1], sgd_3d_np[:, 2],
    color="#4C78A8", linewidth=0.8, alpha=0.38, label="Raw minibatch SGD",
)
ax.plot(
    avg_3d_np[:, 0], avg_3d_np[:, 1], avg_3d_np[:, 2],
    color="#F58518", linewidth=2.2, alpha=0.95, label="Trailing average (window 50)",
)
ax.scatter(
    avg_3d_np[milestones, 0], avg_3d_np[milestones, 1], avg_3d_np[milestones, 2],
    color="#F58518", s=14, alpha=0.9, depthshade=False,
)
ax.scatter(*avg_3d_np[0], color="#54A24B", s=70, marker="o", label="Start", depthshade=False)
ax.scatter(*avg_3d_np[-1], color="#E45756", s=80, marker="X", label="End", depthshade=False)
ax.set_xlabel(pc_label(0), labelpad=10)
ax.set_ylabel(pc_label(1), labelpad=10)
ax.set_zlabel(pc_label(2), labelpad=10)
ax.set_title("Raw SGD vs. trailing-average trajectory: top-3 PCA")
ax.view_init(elev=24, azim=42)
ax.legend(loc="best", frameon=True)
fig.tight_layout()
path_3d = OUT_DIR / "trajectory_pca_3d.png"
fig.savefig(path_3d, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot(
    sgd_2d_np[:, 0], sgd_2d_np[:, 1],
    color="#4C78A8", linewidth=0.8, alpha=0.4, label="Raw minibatch SGD",
)
ax.plot(
    avg_2d_np[:, 0], avg_2d_np[:, 1],
    color="#F58518", linewidth=2.2, alpha=0.95, label="Trailing average (window 50)",
)
ax.scatter(
    avg_2d_np[milestones, 0], avg_2d_np[milestones, 1],
    color="#F58518", s=18, alpha=0.9, zorder=3,
)
ax.scatter(*avg_2d_np[0], color="#54A24B", s=75, marker="o", label="Start", zorder=4)
ax.scatter(*avg_2d_np[-1], color="#E45756", s=85, marker="X", label="End", zorder=4)
ax.set_xlabel(pc_label(0))
ax.set_ylabel(pc_label(1))
ax.set_title("Raw SGD vs. trailing-average trajectory: top-2 PCA")
ax.grid(alpha=0.25)
ax.legend(frameon=True)
fig.tight_layout()
path_2d = OUT_DIR / "trajectory_pca_2d.png"
fig.savefig(path_2d, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 4.8))
loss_smoothing = 25
smoothed_loss = np.convolve(
    losses_np, np.ones(loss_smoothing) / loss_smoothing, mode="valid"
)
ax.plot(np.arange(STEPS), losses_np, color="#4C78A8", alpha=0.25, linewidth=0.8, label="Minibatch loss")
ax.plot(
    np.arange(loss_smoothing - 1, STEPS), smoothed_loss,
    color="#E45756", linewidth=2.0, label=f"{loss_smoothing}-step mean",
)
ax.set(xlabel="SGD update", ylabel="Cross-entropy loss", title="Toy-task training loss")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
path_loss = OUT_DIR / "training_loss.png"
fig.savefig(path_loss, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.8))
shown_components = min(10, evr_np.size)
positions = np.arange(1, shown_components + 1)
ax.bar(positions, 100.0 * evr_np[:shown_components], color="#72B7B2")
ax.set_xticks(positions)
ax.set(
    xlabel="Principal component",
    ylabel="Explained variance (%)",
    title="PCA spectrum of the trailing-average path",
)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
path_variance = OUT_DIR / "pca_explained_variance.png"
fig.savefig(path_variance, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

output_paths = [path_3d, path_2d, path_loss, path_variance]
for output_path in output_paths:
    print(f"Saved {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")